# 🌊 Flood Risk Model — Module 1
**Multi-Hazard Climate Risk System for Indian Real Estate**

This notebook builds a flood risk model for 15 Indian cities using:
- Daily precipitation from Open-Meteo Archive API (2015–2023)
- Daily river discharge from Open-Meteo Flood API (2015–2023)
- Elevation from OpenTopoData SRTM30m
- XGBoost classification for flood probability scoring

In [ ]:
import time
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import os

os.makedirs('../data/outputs', exist_ok=True)

CITIES = [
    {"name": "Jodhpur",    "lat": 26.24, "lon": 73.02},
    {"name": "Jaisalmer",  "lat": 26.92, "lon": 70.91},
    {"name": "Bikaner",    "lat": 28.02, "lon": 73.31},
    {"name": "Nagpur",     "lat": 21.15, "lon": 79.09},
    {"name": "Pune",       "lat": 18.52, "lon": 73.86},
    {"name": "Chennai",    "lat": 13.08, "lon": 80.27},
    {"name": "Bengaluru",  "lat": 12.97, "lon": 77.59},
    {"name": "Hyderabad",  "lat": 17.39, "lon": 78.49},
    {"name": "Bhopal",     "lat": 23.26, "lon": 77.41},
    {"name": "Patna",      "lat": 25.59, "lon": 85.14},
    {"name": "Mumbai",     "lat": 19.08, "lon": 72.88},
    {"name": "Kolkata",    "lat": 22.57, "lon": 88.36},
    {"name": "Guwahati",   "lat": 26.14, "lon": 91.74},
    {"name": "Varanasi",   "lat": 25.32, "lon": 82.97},
    {"name": "Srinagar",   "lat": 34.08, "lon": 74.80},
]

print(f"Configured {len(CITIES)} cities")

## Cell 1 — Fetch Rainfall Data
Daily precipitation (mm) from Open-Meteo Archive API for 2015–2023.

In [ ]:
frames = []
for c in CITIES:
    print(f"Fetching rainfall for {c['name']}...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": c["lat"],
        "longitude": c["lon"],
        "start_date": "2015-01-01",
        "end_date": "2023-12-31",
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    df = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "rainfall_mm": data["daily"]["precipitation_sum"],
        "city": c["name"],
        "lat": c["lat"],
        "lon": c["lon"],
    })
    frames.append(df)
    time.sleep(0.3)

df_rain = pd.concat(frames, ignore_index=True)
df_rain["rainfall_mm"] = df_rain["rainfall_mm"].fillna(0)
print(f"\nRainfall data shape: {df_rain.shape}")
df_rain.head()

## Cell 2 — Fetch River Discharge Data
Daily river discharge (m³/s) from Open-Meteo Flood API for 2015–2023.

In [ ]:
frames = []
for c in CITIES:
    print(f"Fetching river discharge for {c['name']}...")
    url = "https://flood-api.open-meteo.com/v1/flood"
    params = {
        "latitude": c["lat"],
        "longitude": c["lon"],
        "start_date": "2015-01-01",
        "end_date": "2023-12-31",
        "daily": "river_discharge",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    df = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "river_discharge_m3s": data["daily"]["river_discharge"],
        "city": c["name"],
    })
    frames.append(df)
    time.sleep(0.3)

df_flood = pd.concat(frames, ignore_index=True)
df_flood["river_discharge_m3s"] = df_flood["river_discharge_m3s"].fillna(0)
print(f"\nRiver discharge data shape: {df_flood.shape}")
df_flood.head()

## Cell 3 — Fetch Elevation Data
SRTM 30m elevation for all 15 cities in a single batch request.

In [ ]:
locations = "|".join(f"{c['lat']},{c['lon']}" for c in CITIES)
url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations}"
print("Fetching elevation for all cities...")
resp = requests.get(url, timeout=60)
resp.raise_for_status()
data = resp.json()

elevation_dict = {}
for c, result in zip(CITIES, data["results"]):
    elev = result.get("elevation", 0) or 0
    elevation_dict[c["name"]] = elev

print("\nElevation (m):")
for city, elev in elevation_dict.items():
    print(f"  {city}: {elev}")

## Cell 4 — Feature Engineering (Yearly Aggregation)
Merge rainfall + discharge, compute per-city-year features:
- `max_3day_rainfall_mm` — max rolling 3-day precipitation sum
- `days_above_50mm` / `days_above_100mm` — extreme rainfall day counts
- `peak_river_discharge_m3s` — annual peak discharge
- `elevation_m` — city elevation

In [ ]:
df = df_rain.merge(df_flood, on=["city", "date"], how="left")
df["river_discharge_m3s"] = df["river_discharge_m3s"].fillna(0)
df["year"] = df["date"].dt.year

records = []
for (city, year), grp in df.groupby(["city", "year"]):
    grp = grp.sort_values("date")
    rolling_3d = grp["rainfall_mm"].rolling(3).sum()
    max_3day = rolling_3d.max()
    days_50 = (grp["rainfall_mm"] > 50).sum()
    days_100 = (grp["rainfall_mm"] > 100).sum()
    peak_discharge = grp["river_discharge_m3s"].max()
    elev = elevation_dict.get(city, 0)
    lat = grp["lat"].iloc[0]
    lon = grp["lon"].iloc[0]
    records.append({
        "city": city,
        "year": year,
        "lat": lat,
        "lon": lon,
        "max_3day_rainfall_mm": max_3day,
        "days_above_50mm": days_50,
        "days_above_100mm": days_100,
        "peak_river_discharge_m3s": peak_discharge,
        "elevation_m": elev,
    })

df_features = pd.DataFrame(records)
print(f"Feature matrix shape: {df_features.shape}")
df_features.head(10)

## Cell 5 — Flood Labels
- Per city: flood_label = 1 if peak discharge ≥ 80th percentile
- Fallback for all-zero discharge cities: `days_above_100mm > 5`

In [ ]:
df_features["flood_label"] = 0

for city, grp in df_features.groupby("city"):
    idx = grp.index
    if grp["peak_river_discharge_m3s"].sum() == 0:
        # All-zero discharge — use rainfall fallback
        df_features.loc[idx, "flood_label"] = (
            grp["days_above_100mm"] > 5
        ).astype(int).values
    else:
        p80 = grp["peak_river_discharge_m3s"].quantile(0.80)
        df_features.loc[idx, "flood_label"] = (
            grp["peak_river_discharge_m3s"] >= p80
        ).astype(int).values

print("Flood label distribution:")
print(df_features["flood_label"].value_counts())
print(f"\nLabels per city:")
print(df_features.groupby("city")["flood_label"].sum().to_string())

## Cell 6 — Train XGBoost Model
- Train: 2015–2021, Test: 2022–2023
- Features: max_3day_rainfall, days_above_50/100mm, peak_discharge, elevation
- Evaluate: accuracy, AUC-ROC, precision, recall, F1

In [ ]:
FEATURE_COLS = [
    "max_3day_rainfall_mm",
    "days_above_50mm",
    "days_above_100mm",
    "peak_river_discharge_m3s",
    "elevation_m",
]

train = df_features[df_features["year"].between(2015, 2021)]
test = df_features[df_features["year"].between(2022, 2023)]

X_train, y_train = train[FEATURE_COLS], train["flood_label"]
X_test, y_test = test[FEATURE_COLS], test["flood_label"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f"\n{'='*40}")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
try:
    auc = roc_auc_score(y_test, y_proba[:, 1])
    print(f"AUC-ROC  : {auc:.4f}")
except ValueError:
    print("AUC-ROC  : N/A (single class in test set)")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1       : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"{'='*40}")

# Feature importance plot
fig, ax = plt.subplots(figsize=(8, 5))
importance = model.feature_importances_
sorted_idx = np.argsort(importance)
ax.barh([FEATURE_COLS[i] for i in sorted_idx], importance[sorted_idx], color="#2196F3")
ax.set_xlabel("Importance")
ax.set_title("Flood Model — Feature Importance")
plt.tight_layout()
fig.savefig("../data/outputs/flood_feature_importance.png", dpi=150)
plt.show()
print("Saved feature importance plot.")

## Cell 7 — Risk Scores
Compute flood probability → risk score (0–100) → risk category.

In [ ]:
X_all = df_features[FEATURE_COLS]
proba = model.predict_proba(X_all)[:, 1]

risk_scores = df_features[["city", "year", "lat", "lon", "flood_label"]].copy()
risk_scores["flood_prob"] = np.round(proba, 4)
risk_scores["flood_risk_score"] = np.round(proba * 100, 2)

def categorize(score):
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

risk_scores["risk_category"] = risk_scores["flood_risk_score"].apply(categorize)

print("Risk category distribution:")
print(risk_scores["risk_category"].value_counts())
print(f"\nSample scores:")
risk_scores.head(15)

## Cell 8 — Save Outputs
- `flood_risk_scores.csv` — per-city, per-year risk scores
- `flood_model.pkl` — pickled XGBoost model

In [ ]:
risk_scores.to_csv("../data/outputs/flood_risk_scores.csv", index=False)
print("Saved flood_risk_scores.csv")

with open("../data/outputs/flood_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved flood_model.pkl")

print(f"\n{'='*60}")
print("FLOOD RISK MODULE COMPLETE")
print(f"{'='*60}")